In [1]:
# Cell 1 (Bản nâng cấp chạy mọi nơi)
import os
import pandas as pd
from pathlib import Path

# 1. Kiểm tra nếu đang chạy trên Kaggle
if os.path.exists('/kaggle/input'):
    # Thay 'ten-dataset-cua-ban' bằng tên dataset bạn tạo trên Kaggle
    DB_DIR = Path('/kaggle/input/competitions/datathon-2026-round-1/')
    print("🚀 Đang chạy trên Kaggle")
else:
    # 2. Nếu chạy trên máy cá nhân hoặc GitHub
    BASE_DIR = Path.cwd().parent 
    DB_DIR = BASE_DIR / "database"
    print("💻 Đang chạy trên máy cá nhân/GitHub")

def read_data(filename, **kwargs):
    return pd.read_csv(DB_DIR / filename, **kwargs)

print(f"📂 Thư mục dữ liệu hiện tại: {DB_DIR}")


💻 Đang chạy trên máy cá nhân/GitHub
📂 Thư mục dữ liệu hiện tại: /Users/tranquoctuan18112005/Documents/DATATHON/datathon-2026-round-1/bốn con cừu/database


In [2]:
# Q1. Trung vị số ngày giữa hai lần mua liên tiếp
df = read_data('orders.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values(by=['customer_id', 'order_date'])
df['prev_order_date'] = df.groupby('customer_id')['order_date'].shift(1)
df['gap_days'] = (df['order_date'] - df['prev_order_date']).dt.days
median_gap = df['gap_days'].dropna().median()
print(f"🎯 Median Inter-order gap: {median_gap} ngày")


🎯 Median Inter-order gap: 144.0 ngày


In [3]:
# Q2. Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất
df = read_data('products.csv')
df['gross_margin'] = (df['price'] - df['cogs']) / df['price']
top_segment = df.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(f"🎯 Phân khúc lợi nhuận cao nhất: {top_segment.index[0]}")
print(top_segment.head(3))


🎯 Phân khúc lợi nhuận cao nhất: Standard
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Name: gross_margin, dtype: float64


In [4]:
# Q3. Lý do trả hàng Streetwear phổ biến nhất
rets = read_data('returns.csv')
prods = read_data('products.csv')
df = pd.merge(rets, prods, on='product_id', how='inner')
sw_reasons = df[df['category'] == 'Streetwear']['return_reason'].value_counts()
print(f"🎯 Lý do trả hàng Streetwear nhiều nhất: {sw_reasons.index[0]} ({sw_reasons.iloc[0]} lần)")


🎯 Lý do trả hàng Streetwear nhiều nhất: wrong_size (7626 lần)


In [5]:
# Q4. Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất
df = read_data('web_traffic.csv')
lowest_bounce = df.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(f"🎯 Nguồn có Bounce Rate thấp nhất: {lowest_bounce.index[0]} ({lowest_bounce.iloc[0]:.6f})")


🎯 Nguồn có Bounce Rate thấp nhất: email_campaign (0.004458)


In [6]:
# Q5. Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi
df = read_data('order_items.csv', usecols=['promo_id'])
percentage = (df['promo_id'].notnull().sum() / len(df)) * 100
print(f"🎯 Tỷ lệ có áp dụng khuyến mãi: {percentage:.2f}%")


🎯 Tỷ lệ có áp dụng khuyến mãi: 38.66%


In [7]:
# Q6. Nhóm tuổi có số đơn hàng trung bình cao nhất
cust = read_data('customers.csv')
ords = read_data('orders.csv')
cust = cust[cust['age_group'].notnull()]
df = pd.merge(ords, cust, on='customer_id', how='inner')
avg_orders = (df.groupby('age_group')['order_id'].nunique() / cust.groupby('age_group')['customer_id'].nunique()).sort_values(ascending=False)
print(f"🎯 Nhóm tuổi có lượng đơn/người cao nhất: {avg_orders.index[0]} ({avg_orders.iloc[0]:.3f})")


🎯 Nhóm tuổi có lượng đơn/người cao nhất: 55+ (5.407)


In [8]:
# Q7. Vùng tạo ra tổng doanh thu cao nhất
ords = read_data('orders.csv')
items = read_data('order_items.csv', low_memory=False)
geo = read_data('geography.csv')
df = pd.merge(items, ords[['order_id', 'zip']], on='order_id', how='inner')
df = pd.merge(df, geo[['zip', 'region']], on='zip', how='inner')
df['revenue'] = df['unit_price'] * df['quantity'] - df['discount_amount'].fillna(0)
top_region = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
print(f"🎯 Vùng có doanh thu cao nhất: {top_region.index[0]}")


🎯 Vùng có doanh thu cao nhất: East


In [9]:
# Q8. Phương thức thanh toán của đơn hàng 'cancelled' phổ biến nhất
df = read_data('orders.csv', usecols=['order_status', 'payment_method'])
cancelled = df[df['order_status'].str.lower() == 'cancelled']
top_payment = cancelled['payment_method'].value_counts().index[0]
print(f"🎯 Phương thức bị Cancelled nhiều nhất: {top_payment}")


🎯 Phương thức bị Cancelled nhiều nhất: credit_card


In [10]:
# Q9. Kích thước (S, M, L, XL) có tỷ lệ trả hàng cao nhất
prods = read_data('products.csv')
rets = read_data('returns.csv')
items = read_data('order_items.csv', usecols=['product_id'])
size_items = pd.merge(items, prods[['product_id', 'size']], on='product_id')['size'].value_counts()
size_rets = pd.merge(rets, prods[['product_id', 'size']], on='product_id')['size'].value_counts()
rate = (size_rets / size_items * 100).sort_values(ascending=False)
print(f"🎯 Size có tỷ lệ trả hàng cao nhất: {rate.index[0]} ({rate.iloc[0]:.2f}%)")


🎯 Size có tỷ lệ trả hàng cao nhất: S (5.65%)


In [11]:
# Q10. Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất
df = read_data('payments.csv')
top_install = df.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(f"🎯 Kỳ hạn trả góp có giá trị TB cao nhất: {top_install.index[0]} kỳ ({top_install.iloc[0]:,.0f})")


🎯 Kỳ hạn trả góp có giá trị TB cao nhất: 6 kỳ (24,447)
